 # Optuna Hyperparameter Optimisation

 ## Embodiment-Aware Language-to-Command Architecture



 Architecture:



 Description vectors -> Score Network

                    -> Value Network



 Score + Value -> Embodiment latent representation



 Sentence embedding + Embodiment latent -> Main Network -> [x, y, yaw]



 Optuna tunes:

 - Score network architecture

 - Value network architecture

 - Main network architecture

 - Dropout

 - Activation functions

 - Optimizer

 - Learning rate

 - Embodiment latent dimension

In [2]:
# %%
from pathlib import Path
import sys
import csv
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import optuna

from semanticreasoning import ProcessData



ImportError: attempted relative import with no known parent package

 ## Configuration

In [ ]:
# %%
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

EMBEDDING_SIZE = 384
DESCRIPTION_SIZE = 23
OUTPUT_SIZE = 3

BATCH_SIZE = 64

# You can increase this later.
# For Optuna, I strongly recommend NOT starting with 1000.
EPOCHS = 300

N_TRIALS = 100

TRAIN_FILE = "training_data_3_outputs.csv"

# Change this to your actual validation file.
VALIDATION_FILE = "validation_training_data_3_outputs.csv"

ROBOTS = [
    "AnymalB",
    "AnymalC",
    "Badger",
    "UnitreeA1",
    "UnitreeGo1",
    "UnitreeGo2",
    "UnitreeH1",
    "RobotisOP3",
    "Cassie"
]



 ## Load Semantic Embedding Model

In [ ]:
# %%
processor = ProcessData()



 ## Load Robot Description Vectors



 Each robot has multiple 23-dimensional description vectors.



 Shape for one robot:



 ```

 [number_of_joints, 23]

 ```

In [ ]:
# %%
desc_vector_dict = {}

for robot in ROBOTS:

    file_path = f"description_vectors/{robot}_description_vectors.csv"

    with open(file_path, "r") as file:

        reader = csv.reader(file)
        next(reader)

        vectors = []

        for row in reader:

            vector = [
                float(value)
                for value in row[1:]
            ]

            vectors.append(vector)

    desc_vector_dict[robot] = torch.tensor(
        np.array(vectors, dtype=np.float32),
        dtype=torch.float32
    )

    print(
        robot,
        desc_vector_dict[robot].shape
    )



 ## Pre-compute Sentence Embeddings



 This is important.



 Do NOT call the sentence transformer during every Optuna trial.

 Otherwise the exact same sentences would be encoded repeatedly,

 which would make optimisation unnecessarily slow.

In [ ]:
# %%
def load_dataset(file_path):

    texts = []
    robot_names = []
    targets = []

    with open(file_path, "r") as file:

        reader = csv.reader(file)
        next(reader)

        for row in reader:

            if not any(row):
                continue

            text = row[0]
            robot = row[1]

            target = [
                float(value)
                for value in row[2:5]
            ]

            texts.append(text)
            robot_names.append(robot)
            targets.append(target)

    print(f"Encoding {len(texts)} sentences from {file_path}")

    embeddings = processor.getEmbeddings(texts)

    embeddings = torch.tensor(
        np.asarray(embeddings),
        dtype=torch.float32
    )

    targets = torch.tensor(
        targets,
        dtype=torch.float32
    )

    return embeddings, robot_names, targets



 ## Load Training Set

In [ ]:
# %%
train_embeddings, train_robots, train_targets = load_dataset(
    TRAIN_FILE
)

print("Training embeddings:", train_embeddings.shape)
print("Training targets:", train_targets.shape)
print("Training samples:", len(train_robots))



 ## Load Validation Set



 Ideally these should be sentences that are not present in the training set.

In [ ]:
# %%
valid_embeddings, valid_robots, valid_targets = load_dataset(
    VALIDATION_FILE
)

print("Validation embeddings:", valid_embeddings.shape)
print("Validation targets:", valid_targets.shape)
print("Validation samples:", len(valid_robots))



 ## Custom Dataset



 A normal TensorDataset is slightly inconvenient because the robot name

 is a string. This Dataset keeps:



 - sentence embedding

 - robot name

 - target command

In [ ]:
# %%
class EmbodimentDataset(Dataset):

    def __init__(
        self,
        embeddings,
        robot_names,
        targets
    ):

        self.embeddings = embeddings
        self.robot_names = robot_names
        self.targets = targets


    def __len__(self):

        return len(self.targets)


    def __getitem__(self, index):

        return (
            self.embeddings[index],
            self.robot_names[index],
            self.targets[index]
        )



In [ ]:
# %%
train_dataset = EmbodimentDataset(
    train_embeddings,
    train_robots,
    train_targets
)

valid_dataset = EmbodimentDataset(
    valid_embeddings,
    valid_robots,
    valid_targets
)



In [ ]:
# %%
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)



 # Neural Network Construction Helpers

In [ ]:
# %%
def get_activation(name):

    if name == "ReLU":
        return nn.ReLU()

    elif name == "ELU":
        return nn.ELU()

    elif name == "Tanh":
        return nn.Tanh()

    elif name == "LeakyReLU":
        return nn.LeakyReLU()

    elif name == "SiLU":
        return nn.SiLU()

    else:
        raise ValueError(
            f"Unknown activation: {name}"
        )



 ## Score Network



 Original:



 ```python

 23 -> 100 -> 23

 ```



 Instead of forcing its output to remain 23-D, we allow Optuna to choose

 an embodiment latent dimension.



 Therefore:



 ```

 23 -> hidden layers -> latent_dim

 ```



 Each description vector receives a score vector of size `latent_dim`.

In [ ]:
# %%
def define_score_net(trial, latent_dim):

    n_layers = trial.suggest_int(
        "score_n_layers",
        1,
        3
    )

    activation_name = trial.suggest_categorical(
        "score_activation",
        [
            "ReLU",
            "ELU",
            "Tanh",
            "SiLU"
        ]
    )

    layers = []

    in_features = DESCRIPTION_SIZE

    for i in range(n_layers):

        out_features = trial.suggest_int(
            f"score_units_{i}",
            16,
            256,
            log=True
        )

        layers.append(
            nn.Linear(
                in_features,
                out_features
            )
        )

        layers.append(
            get_activation(activation_name)
        )

        dropout = trial.suggest_float(
            f"score_dropout_{i}",
            0.0,
            0.4
        )

        if dropout > 0:
            layers.append(
                nn.Dropout(dropout)
            )

        in_features = out_features

    layers.append(
        nn.Linear(
            in_features,
            latent_dim
        )
    )

    return nn.Sequential(*layers)



 ## Value Network



 Produces the value representation corresponding to the score network.



 ```

 23 -> hidden layers -> latent_dim

 ```

In [ ]:
# %%
def define_value_net(trial, latent_dim):

    n_layers = trial.suggest_int(
        "value_n_layers",
        1,
        3
    )

    activation_name = trial.suggest_categorical(
        "value_activation",
        [
            "ReLU",
            "ELU",
            "Tanh",
            "SiLU"
        ]
    )

    layers = []

    in_features = DESCRIPTION_SIZE

    for i in range(n_layers):

        out_features = trial.suggest_int(
            f"value_units_{i}",
            16,
            256,
            log=True
        )

        layers.append(
            nn.Linear(
                in_features,
                out_features
            )
        )

        layers.append(
            get_activation(activation_name)
        )

        dropout = trial.suggest_float(
            f"value_dropout_{i}",
            0.0,
            0.4
        )

        if dropout > 0:
            layers.append(
                nn.Dropout(dropout)
            )

        in_features = out_features

    layers.append(
        nn.Linear(
            in_features,
            latent_dim
        )
    )

    return nn.Sequential(*layers)



 ## Main Network



 Input:



 ```

 sentence embedding + embodiment latent

 ```



 Therefore:



 ```

 input size = 384 + latent_dim

 ```



 Output:



 ```

 [x, y, yaw]

 ```

In [ ]:
# %%
def define_main_net(
    trial,
    latent_dim
):

    input_size = (
        EMBEDDING_SIZE
        + latent_dim
    )

    n_layers = trial.suggest_int(
        "main_n_layers",
        1,
        4
    )

    activation_name = trial.suggest_categorical(
        "main_activation",
        [
            "ReLU",
            "ELU",
            "Tanh",
            "SiLU"
        ]
    )

    layers = []

    in_features = input_size

    for i in range(n_layers):

        out_features = trial.suggest_int(
            f"main_units_{i}",
            16,
            512,
            log=True
        )

        layers.append(
            nn.Linear(
                in_features,
                out_features
            )
        )

        layers.append(
            get_activation(
                activation_name
            )
        )

        dropout = trial.suggest_float(
            f"main_dropout_{i}",
            0.0,
            0.5
        )

        if dropout > 0:

            layers.append(
                nn.Dropout(dropout)
            )

        in_features = out_features

    layers.append(
        nn.Linear(
            in_features,
            OUTPUT_SIZE
        )
    )

    return nn.Sequential(*layers)



 # Embodiment Latent Calculation



 This reproduces the important part of your current architecture:



 ```python

 scores = scores_net(description)

 weights = softmax(scores)



 values = value_net(description)



 latent = values * weights

 latent_sum = latent.sum(dim=0)

 ```
 

In [ ]:
# %%
def calculate_robot_latent(
    robot_name,
    score_net,
    value_net
):

    description = desc_vector_dict[
        robot_name
    ].to(DEVICE)

    scores = score_net(
        description
    )

    weights = torch.softmax(
        scores,
        dim=-1
    )

    values = value_net(
        description
    )

    latent = (
        values
        * weights
    )

    latent_sum = latent.sum(
        dim=0
    )

    return latent_sum



 ## Batch Latent Calculation

In [ ]:
# %%
def calculate_batch_latents(
    robot_names,
    score_net,
    value_net
):

    latent_list = []

    for robot in robot_names:

        latent = calculate_robot_latent(
            robot,
            score_net,
            value_net
        )

        latent_list.append(
            latent
        )

    return torch.stack(
        latent_list
    )



 # Optuna Objective

In [ ]:
# %%
def objective(trial):

    # -------------------------------------------
    # Embodiment latent size
    # -------------------------------------------

    latent_dim = 23

    # -------------------------------------------
    # Create networks
    # -------------------------------------------

    score_net = define_score_net(
        trial,
        latent_dim
    ).to(DEVICE)

    value_net = define_value_net(
        trial,
        latent_dim
    ).to(DEVICE)

    main_net = define_main_net(
        trial,
        latent_dim
    ).to(DEVICE)

    # -------------------------------------------
    # Optimizer
    # -------------------------------------------

    optimizer_name = (
        trial.suggest_categorical(
            "optimizer",
            [
                "Adam",
                "AdamW",
                "RMSprop"
            ]
        )
    )

    learning_rate = (
        trial.suggest_float(
            "lr",
            1e-5,
            5e-3,
            log=True
        )
    )

    # All three networks MUST be optimized
    # together because the latent representation
    # is learned jointly with command prediction.

    parameters = (
        list(score_net.parameters())
        + list(value_net.parameters())
        + list(main_net.parameters())
    )

    if optimizer_name == "Adam":

        optimizer = optim.Adam(
            parameters,
            lr=learning_rate
        )

    elif optimizer_name == "AdamW":

        weight_decay = (
            trial.suggest_float(
                "weight_decay",
                1e-6,
                1e-2,
                log=True
            )
        )

        optimizer = optim.AdamW(
            parameters,
            lr=learning_rate,
            weight_decay=weight_decay
        )

    elif optimizer_name == "RMSprop":

        optimizer = optim.RMSprop(
            parameters,
            lr=learning_rate
        )

    else:

        raise ValueError(
            optimizer_name
        )

    loss_fn = nn.MSELoss()

    # -------------------------------------------
    # Training
    # -------------------------------------------

    for epoch in range(EPOCHS):

        score_net.train()
        value_net.train()
        main_net.train()

        training_loss = 0.0

        for (
            embeddings,
            robot_names,
            targets
        ) in train_loader:

            embeddings = embeddings.to(
                DEVICE
            )

            targets = targets.to(
                DEVICE
            )

            # -----------------------------------
            # Calculate embodiment representations
            # -----------------------------------

            latent_batch = (
                calculate_batch_latents(
                    robot_names,
                    score_net,
                    value_net
                )
            )

            # -----------------------------------
            # Combine language and embodiment
            # -----------------------------------

            batch_input = torch.cat(
                [
                    embeddings,
                    latent_batch
                ],
                dim=1
            )

            # -----------------------------------
            # Prediction
            # -----------------------------------

            predictions = main_net(
                batch_input
            )

            loss = loss_fn(
                predictions,
                targets
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            training_loss += (
                loss.item()
            )

        average_training_loss = (
            training_loss
            / len(train_loader)
        )

        # -------------------------------------------
        # Validation
        # -------------------------------------------

        score_net.eval()
        value_net.eval()
        main_net.eval()

        total_squared_error = 0.0
        total_elements = 0

        with torch.no_grad():

            for (
                embeddings,
                robot_names,
                targets
            ) in valid_loader:

                embeddings = embeddings.to(
                    DEVICE
                )

                targets = targets.to(
                    DEVICE
                )

                latent_batch = (
                    calculate_batch_latents(
                        robot_names,
                        score_net,
                        value_net
                    )
                )

                batch_input = torch.cat(
                    [
                        embeddings,
                        latent_batch
                    ],
                    dim=1
                )

                predictions = main_net(
                    batch_input
                )

                squared_error = (
                    predictions
                    - targets
                ).pow(2)

                total_squared_error += (
                    squared_error.sum().item()
                )

                total_elements += (
                    targets.numel()
                )

        validation_mse = (
            total_squared_error
            / total_elements
        )

        # -------------------------------------------
        # Report validation result to Optuna
        # -------------------------------------------

        trial.report(
            validation_mse,
            epoch
        )

        # -------------------------------------------
        # Pruning
        # -------------------------------------------

        if trial.should_prune():

            raise (
                optuna.exceptions
                .TrialPruned()
            )

    return validation_mse



 # Create Optuna Study



 MedianPruner stops trials which perform substantially worse than the

 current median of completed trials.

In [ ]:
# %%
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=30,
        interval_steps=10
    )
)



 ## Run Optimisation

In [ ]:
# %%
study.optimize(
    objective,
    n_trials=N_TRIALS
)



 ## Best Result

In [ ]:
# %%
print(
    "Number of completed trials:",
    len(study.trials)
)

print(
    "\nBest validation MSE:",
    study.best_value
)

print(
    "\nBest parameters:"
)

for key, value in study.best_params.items():

    print(
        f"{key}: {value}"
    )



 ## Best Trial

In [ ]:
# %%
best_trial = study.best_trial

print(
    "Best trial number:",
    best_trial.number
)

print(
    "Best validation MSE:",
    best_trial.value
)



 # Optuna Results DataFrame

In [ ]:
# %%
results_df = (
    study.trials_dataframe()
)

results_df.head()



 ## Save Optuna Results

In [ ]:
# %%
results_df.to_csv(
    "embodiment_optuna_trials.csv",
    index=False
)



 # Visualisation



 These work especially well in a notebook.

In [ ]:
# %%
try:

    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances
    )

    plot_optimization_history(
        study
    ).show()

except Exception as e:

    print(
        "Could not display optimisation history:",
        e
    )



In [ ]:
# %%
try:

    plot_param_importances(
        study
    ).show()

except Exception as e:

    print(
        "Could not display parameter importance:",
        e
    )



 # Rebuild the Best Architecture



 Once Optuna identifies the best hyperparameters, reconstruct the model.



 This alone DOES NOT retain the trained weights from the Optuna trial.

 We will retrain the selected architecture properly afterwards.

In [ ]:
# %%
best_params = (
    study.best_params
)

best_params



 ## FixedTrial



 FixedTrial allows us to reuse exactly the architecture chosen by Optuna.

In [ ]:
# %%
fixed_trial = optuna.trial.FixedTrial(
    best_params
)

best_latent_dim = (
    best_params["latent_dim"]
)

best_score_net = (
    define_score_net(
        fixed_trial,
        best_latent_dim
    ).to(DEVICE)
)

best_value_net = (
    define_value_net(
        fixed_trial,
        best_latent_dim
    ).to(DEVICE)
)

best_main_net = (
    define_main_net(
        fixed_trial,
        best_latent_dim
    ).to(DEVICE)
)



In [ ]:
# %%
print(best_score_net)

print("\nVALUE NETWORK")
print(best_value_net)

print("\nMAIN NETWORK")
print(best_main_net)



 # Train Final Model



 After Optuna selects the architecture, retrain the selected architecture.



 You can now use a larger epoch count than during optimisation.

In [ ]:
# %%
FINAL_EPOCHS = 1000



In [ ]:
# %%
optimizer_name = best_params[
    "optimizer"
]

learning_rate = best_params[
    "lr"
]

parameters = (
    list(best_score_net.parameters())
    + list(best_value_net.parameters())
    + list(best_main_net.parameters())
)


if optimizer_name == "Adam":

    final_optimizer = optim.Adam(
        parameters,
        lr=learning_rate
    )


elif optimizer_name == "AdamW":

    final_optimizer = optim.AdamW(
        parameters,
        lr=learning_rate,
        weight_decay=best_params[
            "weight_decay"
        ]
    )


elif optimizer_name == "RMSprop":

    final_optimizer = optim.RMSprop(
        parameters,
        lr=learning_rate
    )


loss_fn = nn.MSELoss()



In [ ]:
# %%
final_training_history = []
final_validation_history = []

best_validation_loss = float("inf")

best_score_state = None
best_value_state = None
best_main_state = None


for epoch in range(FINAL_EPOCHS):

    # ==========================================
    # TRAIN
    # ==========================================

    best_score_net.train()
    best_value_net.train()
    best_main_net.train()

    total_training_loss = 0.0

    for (
        embeddings,
        robot_names,
        targets
    ) in train_loader:

        embeddings = embeddings.to(
            DEVICE
        )

        targets = targets.to(
            DEVICE
        )

        latent_batch = (
            calculate_batch_latents(
                robot_names,
                best_score_net,
                best_value_net
            )
        )

        model_input = torch.cat(
            [
                embeddings,
                latent_batch
            ],
            dim=1
        )

        predictions = (
            best_main_net(
                model_input
            )
        )

        loss = loss_fn(
            predictions,
            targets
        )

        final_optimizer.zero_grad()

        loss.backward()

        final_optimizer.step()

        total_training_loss += (
            loss.item()
        )

    average_training_loss = (
        total_training_loss
        / len(train_loader)
    )

    # ==========================================
    # VALIDATE
    # ==========================================

    best_score_net.eval()
    best_value_net.eval()
    best_main_net.eval()

    total_squared_error = 0.0
    total_elements = 0

    with torch.no_grad():

        for (
            embeddings,
            robot_names,
            targets
        ) in valid_loader:

            embeddings = embeddings.to(
                DEVICE
            )

            targets = targets.to(
                DEVICE
            )

            latent_batch = (
                calculate_batch_latents(
                    robot_names,
                    best_score_net,
                    best_value_net
                )
            )

            model_input = torch.cat(
                [
                    embeddings,
                    latent_batch
                ],
                dim=1
            )

            predictions = (
                best_main_net(
                    model_input
                )
            )

            squared_error = (
                predictions
                - targets
            ).pow(2)

            total_squared_error += (
                squared_error.sum().item()
            )

            total_elements += (
                targets.numel()
            )

    validation_mse = (
        total_squared_error
        / total_elements
    )

    final_training_history.append(
        average_training_loss
    )

    final_validation_history.append(
        validation_mse
    )

    # ==========================================
    # SAVE BEST WEIGHTS IN MEMORY
    # ==========================================

    if validation_mse < best_validation_loss:

        best_validation_loss = (
            validation_mse
        )

        best_score_state = copy.deepcopy(
            best_score_net.state_dict()
        )

        best_value_state = copy.deepcopy(
            best_value_net.state_dict()
        )

        best_main_state = copy.deepcopy(
            best_main_net.state_dict()
        )

    # ==========================================
    # PRINT
    # ==========================================

    if (
        epoch == 0
        or (epoch + 1) % 50 == 0
    ):

        print(
            f"Epoch [{epoch + 1}/{FINAL_EPOCHS}] "
            f"Train MSE: {average_training_loss:.6f} | "
            f"Validation MSE: {validation_mse:.6f}"
        )



 ## Restore Best Validation Checkpoint



 The final epoch is not necessarily the best epoch.

In [ ]:
# %%
best_score_net.load_state_dict(
    best_score_state
)

best_value_net.load_state_dict(
    best_value_state
)

best_main_net.load_state_dict(
    best_main_state
)

print(
    "Best final validation MSE:",
    best_validation_loss
)



 # Training Curves

In [ ]:
# %%
import matplotlib.pyplot as plt



In [ ]:
# %%
plt.figure()

plt.plot(
    final_training_history,
    label="Training MSE"
)

plt.plot(
    final_validation_history,
    label="Validation MSE"
)

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title(
    "Embodiment-Aware Model Training"
)

plt.legend()
plt.grid()

plt.show()



 # Test the Optimised Architecture

In [ ]:
# %%
def predict_command(
    text,
    robot_name
):

    best_score_net.eval()
    best_value_net.eval()
    best_main_net.eval()

    embedding = (
        processor.getEmbeddings(
            text
        )
    )

    embedding = torch.tensor(
        embedding,
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():

        latent = (
            calculate_robot_latent(
                robot_name,
                best_score_net,
                best_value_net
            )
        )

        model_input = torch.cat(
            [
                embedding,
                latent
            ]
        )

        output = best_main_net(
            model_input
        )

    return output.cpu()



In [ ]:
# %%
test_output = predict_command(
    "move forward",
    "UnitreeGo1"
)

print(
    "Predicted command:",
    test_output
)



 # Save Final Networks

In [ ]:
# %%
torch.save(
    best_score_net.state_dict(),
    "optimized_score_nn.pth"
)

torch.save(
    best_value_net.state_dict(),
    "optimized_value_nn.pth"
)

torch.save(
    best_main_net.state_dict(),
    "optimized_main_nn.pth"
)



 ## Save Architecture / Hyperparameters



 This is important because state_dict alone does not contain enough

 information to reconstruct an Optuna-selected variable architecture.

In [ ]:
# %%
import json



In [ ]:
# %%
with open(
    "embodiment_optuna_best_parameters.json",
    "w"
) as file:

    json.dump(
        best_params,
        file,
        indent=4
    )



In [ ]:
# %%
print(
    "Models and Optuna parameters saved."
)